In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")

# if not _os.path.exists(narration_dir):
#     _os.makedirs(narration_dir, exist_ok=True)
try:
    import gdown
    gdown.download(id="1zgBFdCQiB3mJoQ5bHGGCh_8WkdJkxWy0", output=f"{narration_dir}/narration.zip", quiet=False)
    !unzip -q {narration_dir}/narration.zip -d {narration_dir}
    !rm {narration_dir}/narration.zip
    print(f"Loaded {len(_os.listdir(narration_dir))} narration segments")
except Exception as e:
    print(f"⚠️ Could not download narration ({type(e).__name__}: {e})")
    print("This is expected if running locally. Audio narration is optional.")

from IPython.display import Audio, display
_f = f"{narration_dir}/03_00_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
#@title 🎧 Code Walkthrough: Setup And Imports
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_01_setup_and_imports.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [2]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
# elif torch.backends.mps.is_available():
#     device = torch.device('mps')
#     print("✅ Apple Silicon GPU available (Metal Performance Shaders)")

else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
elif torch.backends.mps.is_available():
    torch.manual_seed(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

⚠️ No GPU detected. Some cells may run slowly.
   Go to Runtime → Change runtime type → GPU

📦 Python 3.13.12
🔥 PyTorch 2.11.0
🎲 Random seed set to 42


# Training a Reasoning Model End-to-End -- Vizuara

In this notebook, we combine SFT and GRPO to train a language model that reasons through math problems. This is the full pipeline from base model to reasoning model.

**What you will build:** A model fine-tuned with SFT then GRPO on math problems, producing chain-of-thought reasoning.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
import matplotlib.pyplot as plt
import numpy as np
import re
import copy
%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}, Device: {device}")
torch.manual_seed(42)

PyTorch: 2.11.0, Device: cpu


In [ ]:
#@title 🎧 Listen: Why This Matters
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_02_why_this_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. Why Does This Matter?

In Notebooks 1 and 2, we built individual components: SFT and GRPO. Now we combine them into the full pipeline. By the end, you will see a small model develop step-by-step reasoning ability.

**Teaser output from our trained model:**
```
Q: If a box has 3 rows of 4 apples, remove 5. How many left?
<think> 3 rows of 4 = 3*4 = 12. Remove 5: 12-5 = 7. </think>
The answer is 7.
```

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_03_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building Intuition

Think of two-phase training like teaching a student:
- **Phase 1 (SFT):** Show worked examples. Student learns the *format*.
- **Phase 2 (GRPO):** Practice exams with grading. Student learns *what works*.

Neither alone suffices. SFT without RL = plausible but wrong reasoning. RL without SFT = no idea how to structure reasoning.

### Think About This
- Why must SFT come before RL?
- What if SFT was very long but RL very short?

In [ ]:
#@title 🎧 Listen: The Mathematics
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_04_the_mathematics.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. The Mathematics

**SFT loss:** $\mathcal{L}_{\text{SFT}} = -\sum_{t} \log p_\theta(y_t | y_{<t}, x)$

**GRPO objective:**
$$\mathcal{L}_{\text{GRPO}} = -\frac{1}{G}\sum_{i=1}^{G} \min\left(r_i \hat{A}_i, \text{clip}(r_i, 1-\epsilon, 1+\epsilon) \hat{A}_i\right)$$

where $\hat{A}_i = \frac{R_i - \bar{R}}{\sigma_R}$ and $r_i = \frac{\pi_\theta(y_i|x)}{\pi_{\text{old}}(y_i|x)}$.

**Computationally:** SFT minimizes prediction error on worked examples. GRPO maximizes probability of correct completions relative to the group.

In [ ]:
#@title 🎧 Code Walkthrough: Model And Data Setup
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_05_model_and_data_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Let's Build It -- Component by Component

### 4.1 Setup: Model and Data

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_OPTIONS = {
    "pythia": "EleutherAI/pythia-410m",
    "qwen": "Qwen/Qwen2.5-0.5B",
}

MODEL_KEY = "pythia"  # Change to "qwen" to try Qwen/Qwen2.5-0.5B next.
MODEL_NAME = MODEL_OPTIONS[MODEL_KEY]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

special_tokens = {"additional_special_tokens": ["<think>", "</think>"]}
added = tokenizer.add_special_tokens(special_tokens)
if added > 0:
    model.resize_token_embeddings(len(tokenizer))

print(f"Model: {MODEL_NAME} ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"Model key: {MODEL_KEY}")

/Users/pranavr/Developer/Prep/Courses/visuara/reasoning_model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 292/292 [00:00<00:00, 7352.60it/s]


Model: EleutherAI/pythia-410m (405,282,816 params)
Model key: pythia


In [5]:
from datasets import load_dataset

GSM8K_SFT_LIMIT = 256
SVAMP_GRPO_LIMIT = 256
SVAMP_EVAL_LIMIT = 100

def build_gsm8k_sft_data(limit=256):
    """Load a compact GSM8K SFT set with <think> formatting."""
    ds = load_dataset("openai/gsm8k", "main")
    formatted = []

    for example in ds["train"]:
        parts = example["answer"].rsplit("####", 1)
        if len(parts) != 2:
            continue

        reasoning = re.sub(r"<<[^>]+>>", "", parts[0]).strip()
        final_answer = parts[1].strip().replace(",", "")
        formatted.append(
            (
                example["question"],
                f"<think>\n{reasoning}\n</think>\nThe answer is {final_answer}.",
            )
        )
        if len(formatted) >= limit:
            break

    return formatted

def build_svamp_problem_sets(train_limit=256, test_limit=100):
    """Load SVAMP for verifiable GRPO training and held-out evaluation."""
    ds = load_dataset("ChilleD/SVAMP")

    def normalize(split, limit):
        problems = []
        for example in ds[split]:
            prompt = example.get("question_concat") or f"{example['Body']} {example['Question']}"
            answer = str(example["Answer"]).replace(",", "").strip()
            problems.append((prompt.strip(), answer))
            if len(problems) >= limit:
                break
        return problems

    return normalize("train", train_limit), normalize("test", test_limit)

sft_data = build_gsm8k_sft_data(limit=GSM8K_SFT_LIMIT)
rl_problems, eval_problems = build_svamp_problem_sets(
    train_limit=SVAMP_GRPO_LIMIT,
    test_limit=SVAMP_EVAL_LIMIT,
)

print(f"SFT examples (GSM8K): {len(sft_data)}")
print(f"GRPO problems (SVAMP train): {len(rl_problems)}")
print(f"Eval problems (SVAMP test): {len(eval_problems)}")
print(f"First SFT prompt: {sft_data[0][0]}")
print(f"First GRPO prompt: {rl_problems[0][0]}")

SFT examples (GSM8K): 256
GRPO problems (SVAMP train): 256
Eval problems (SVAMP test): 100
First SFT prompt: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
First GRPO prompt: There are 87 oranges and 290 bananas in Philip's collection. If the bananas are organized into 2 groups and oranges are organized into 93 groups How big is each group of bananas?


In [6]:
def extract_answer(text):
    """Extract the final numerical answer from model output."""
    match = re.search(r'[Tt]he answer is[:\s]*(\-?[\d,\.]+)', text)
    if match:
        return match.group(1).replace(',', '').strip('.')
    numbers = re.findall(r'\-?[\d]+(?:\.\d+)?', text)
    return numbers[-1] if numbers else None


def compute_reward(completion, ground_truth, atol=1e-6):
    """Shaped reward to reduce cold-start failures during GRPO."""
    reward = 0.0

    if "The answer is" in completion:
        reward += 0.1
    if "<think>" in completion and "</think>" in completion:
        reward += 0.1

    predicted = extract_answer(completion)
    if predicted is None:
        return reward

    reward += 0.2
    predicted = predicted.strip().replace(",", "")
    target = ground_truth.strip().replace(",", "")

    try:
        pred_value = float(predicted)
        target_value = float(target)
        error = abs(pred_value - target_value)

        if error <= atol:
            return 1.0
        if target_value != 0:
            rel_error = error / abs(target_value)
            if rel_error <= 0.05:
                reward += 0.5
            elif rel_error <= 0.25:
                reward += 0.25
        elif error <= 1.0:
            reward += 0.25
    except ValueError:
        if predicted == target:
            return 1.0

    return min(reward, 0.95)


# Test
print(compute_reward("<think>\n32.\n</think>\nThe answer is 32.0", "32"))  # Should be 1.0
print(compute_reward("<think>\n30.\n</think>\nThe answer is 0.", "32"))  # Should be > 0 due to shaped reward

1.0
0.4


In [ ]:
#@title 🎧 Code Walkthrough: Sft Training
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_06_sft_training.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 4.2 Phase 1: SFT Training

In [7]:
def run_sft(model, tokenizer, data, epochs=10, minibatch_size=32, lr=5e-6, max_length=256, min_lr=1e-6):
    """Run SFT with one fresh shuffled minibatch per epoch."""
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.0, eps=1e-6)
    losses = []

    if device.type == "mps" and MODEL_KEY == "pythia":
        print("Warning: pythia on MPS can be numerically unstable; CPU or CUDA is safer.")

    for epoch in range(epochs):
        model.train()
        batch_size = min(minibatch_size, len(data))
        batch_indices = np.random.choice(len(data), size=batch_size, replace=False)
        batch_text = [f"Question: {data[i][0]}\n{data[i][1]}" for i in batch_indices]

        tokens = tokenizer(
            batch_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        tokens = {k: v.to(device) for k, v in tokens.items()}
        labels = tokens["input_ids"].clone()
        labels[tokens["attention_mask"] == 0] = -100

        optimizer.zero_grad(set_to_none=True)
        loss = model(**tokens, labels=labels).loss

        if not torch.isfinite(loss):
            new_lr = max(optimizer.param_groups[0]["lr"] * 0.5, min_lr)
            for group in optimizer.param_groups:
                group["lr"] = new_lr
            print(f"SFT Epoch {epoch+1}, non-finite loss; skipping minibatch and reducing lr to {new_lr:.2e}")
            continue

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        if not torch.isfinite(grad_norm):
            optimizer.zero_grad(set_to_none=True)
            new_lr = max(optimizer.param_groups[0]["lr"] * 0.5, min_lr)
            for group in optimizer.param_groups:
                group["lr"] = new_lr
            print(f"SFT Epoch {epoch+1}, non-finite gradients; skipping minibatch and reducing lr to {new_lr:.2e}")
            continue

        optimizer.step()

        losses.append(loss.item())
        print(f"SFT Epoch {epoch+1}, Loss: {losses[-1]:.4f}, LR: {optimizer.param_groups[0]['lr']:.2e}")

    return losses

print("Phase 1: SFT Training")
sft_losses = run_sft(model, tokenizer, sft_data)

Phase 1: SFT Training


: 

In [ ]:
#@title 🎧 What to Look For: Sft Loss Visualization
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_07_sft_loss_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Visualization Checkpoint: SFT Loss

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(sft_losses, color='#2196F3', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('SFT Training Loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
#@title 🎧 Code Walkthrough: Save Reference Model
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_08_save_reference_model.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 4.3 Save Reference Model

In [ ]:
ref_model = copy.deepcopy(model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False
print("Reference model saved (frozen)")

In [ ]:
#@title 🎧 Code Walkthrough: Grpo Helper Functions
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_09_grpo_helper_functions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 4.4 Phase 2: GRPO Training

In [ ]:
def generate(model, tokenizer, prompt, max_new=128, temperature=0.8, top_p=0.95):
    """Generate completion from model."""
    text = f"Question: {prompt}\n"
    inputs = tokenizer(text, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new,
                           do_sample=True, temperature=temperature, top_p=top_p,
                           pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=False)[len(text):], out[0]

def log_prob(model, toks):
    """Compute total log probability of a token sequence with gradients."""
    logits = model(toks.unsqueeze(0).to(device)).logits[0, :-1]
    targets = toks[1:].to(device)
    lp = F.log_softmax(logits, dim=-1)
    return lp.gather(1, targets.unsqueeze(1)).squeeze().sum()

In [ ]:
#@title 🎧 Before You Start: Todo Grpo Loop
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_10_todo_1_grpo_loop.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Your Turn -- TODO Exercises

### TODO 1: Complete the GRPO Training Loop

In [ ]:
def grpo_train(model, ref_model, tokenizer, problems,
               steps=60, G=8, beta=0.02, lr=2e-6,
               temperature=1.0, top_p=0.95, log_every=10):
    """GRPO training loop."""
    optimizer = AdamW(model.parameters(), lr=lr)
    rewards_hist, losses_hist = [], []

    for step in range(steps):
        idx = np.random.randint(len(problems))
        prompt, truth = problems[idx]

        # Generate G completions
        comps, toks_list = [], []
        for _ in range(G):
            c, t = generate(model, tokenizer, prompt, temperature=temperature, top_p=top_p)
            comps.append(c); toks_list.append(t)

        # Compute rewards
        rewards = torch.tensor([compute_reward(c, truth) for c in comps], device=device)

        if rewards.std() < 1e-8:
            advantages = torch.zeros_like(rewards)
        else:
            advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)

        # Compute log probs
        lp_curr = torch.stack([log_prob(model, t) for t in toks_list])
        with torch.no_grad():
            lp_ref = torch.stack([log_prob(ref_model, t) for t in toks_list])

        # KL penalty
        kl = (lp_curr - lp_ref)
        adj_rewards = rewards - beta * kl
        if adj_rewards.std() > 1e-8:
            advantages = (adj_rewards - adj_rewards.mean()) / (adj_rewards.std() + 1e-8)

        loss = -(lp_curr * advantages.to(lp_curr.device)).mean()

        if hasattr(loss, 'backward'):
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        rewards_hist.append(rewards.mean().item())
        losses_hist.append(loss.item())

        if (step + 1) % log_every == 0:
            best_idx = int(rewards.argmax().item())
            print(f"Step {step+1}, Avg Reward: {np.mean(rewards_hist[-log_every:]):.3f}, Best Reward: {rewards[best_idx].item():.3f}")
            print(f"Prompt: {prompt}")
            print(f"Best completion: {comps[best_idx][:200]}\n")

    return rewards_hist

print("Phase 2: GRPO Training")
rl_rewards = grpo_train(model, ref_model, tokenizer, rl_problems)

In [ ]:
#@title 🎧 Narration: After Todo
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_11_after_todo_1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification
assert len(rl_rewards) > 0, "No rewards recorded"
print(f"Training complete! Final avg reward: {np.mean(rl_rewards[-10:]):.3f}")

In [ ]:
#@title 🎧 Before You Start: Todo Compare Outputs
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_12_todo_2_compare_outputs.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Compare Pre-RL vs Post-RL Outputs

In [ ]:
# ============ TODO ============
# Generate from both ref_model and model on rl_problems[:4]
# Compare reasoning quality
# for prompt, truth in rl_problems[:4]:
#     before, _ = generate(ref_model, tokenizer, prompt)
#     after, _ = generate(model, tokenizer, prompt)
#     print(f"Q: {prompt} (Truth: {truth})")
#     print(f"Before: {before[:150]}")
#     print(f"After:  {after[:150]}\n")
# ==============================

In [ ]:
#@title 🎧 Code Walkthrough: Putting It All Together
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_13_putting_it_all_together.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Putting It All Together

In [ ]:
print("=== Final Evaluation (SVAMP Test) ===\n")
correct = 0
for prompt, truth in eval_problems:
    comp, _ = generate(model, tokenizer, prompt)
    r = compute_reward(comp, truth)
    correct += int(r == 1.0)
    print(f"Q: {prompt} | Truth: {truth} | {'CORRECT' if r == 1.0 else 'WRONG'}")
    print(f"  {comp[:200]}\n")
print(f"Accuracy: {correct}/{len(eval_problems)} ({100*correct/len(eval_problems):.0f}%)")

In [ ]:
#@title 🎧 What to Look For: Training And Results Viz
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_14_training_and_results_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Training and Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(sft_losses, color='#2196F3', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Phase 1: SFT'); ax1.grid(True, alpha=0.3)

w = 5
sm = [np.mean(rl_rewards[max(0,i-w):i+1]) for i in range(len(rl_rewards))]
ax2.plot(sm, color='#4CAF50', linewidth=2)
ax2.set_xlabel('Step'); ax2.set_ylabel('Avg Reward')
ax2.set_title('Phase 2: GRPO'); ax2.set_ylim(-0.1, 1.1); ax2.grid(True, alpha=0.3)

plt.suptitle('Complete Training Pipeline', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
#@title 🎧 Listen: Final Output
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_15_final_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Final Output

We trained a reasoning model through the full two-phase pipeline:
1. **SFT:** Learned the `<think>` format from worked examples
2. **GRPO:** Learned which reasoning strategies lead to correct answers

Even with GPT-2 (124M params) and 8 toy problems, we observe the model developing reasoning behavior. At DeepSeek scale (671B params, GSM8K), this produces remarkable emergent reasoning.

In [ ]:
#@title 🎧 Wrap-Up: Reflection And Next Steps
from IPython.display import Audio, display
import os as _os
narration_dir = _os.path.expanduser("~/Developer/Prep/Courses/visuara/reasoning_model/narration")
_f = f"{narration_dir}/03_16_reflection_and_next_steps.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Reflection and Next Steps

### Think About This
1. How would a larger model and dataset change the results?
2. DeepSeek-R1 found RL alone (no SFT) can work. Why?
3. What happens if we increase the group size G from 4 to 64?

### What Comes Next
Notebook 4 covers emergent behaviors (self-verification, backtracking) and distillation from large to small models.

### Key Takeaway
SFT provides scaffolding (format). RL fills it with substance (correctness). Together they produce genuine reasoning.